# Validación del Balanceo Demográfico — v2

Este notebook verifica que los datasets balanceados cumplen las restricciones propias de cada técnica.

**Técnicas actuales y sus restricciones esperadas:**

| Categoría | Técnica | Efecto en train | Duplicados |
|---|---|---|---|
| Aleatoria | `undersampling` (RUS) | Reduce al grupo minoritario | ❌ No crea nuevos |
| Aleatoria | `oversampling` (ROS) | Expande al grupo mayoritario | ✅ Introduce |
| Guiada | `smote` | Expande con interpolación | ≈0 (puntos nuevos) |
| Guiada | `tomek_links` | Elimina pares Tomek (limpieza frontera) | ❌ No crea |
| Combinada | `patient_aware_undersampling` | Reduce respetando cuota/paciente | ❌ No crea |
| Combinada | `jittering` | Expande con ruido gaussiano | ≈0 (puntos nuevos) |
| Combinada | `undersampling_oversampling` | Mismo tamaño, proporción equitativa (RUS+ROS) | Solo si se amplía algún grupo |
| Combinada | `undersampling_smote` | Mismo tamaño, proporción equitativa (RUS+SMOTE) | ≈0 |
| Combinada | `smote_tomek` | Proporción equitativa: SMOTE para crecer, Tomek para reducir | ≈0 |

**Grupos de edad:** `<31`, `31-45`, `46-65`, `>=66`

**Para cada archivo balanceado disponible se verifica:**
1. Efecto correcto sobre el tamaño de train.
2. val y test bit-a-bit idénticos al original.
3. Sin solapamiento de pacientes entre splits.
4. Distribución de grupos antes y después.
5. Verificación específica según la técnica.

> Solo se analizan los archivos disponibles en `balanced_outputs/`. Los folds no presentes se omiten.

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings('ignore')

# ── Rutas ─────────────────────────────────────────────────────────────────────
DATA_ROOT = Path(r'C:/Users/User/Desktop/TFM-Glucose-Prediction/data')

DATASETS = {
    'DIATREND':          DATA_ROOT / 'DIATREND',
    'REPLACE-BG':        DATA_ROOT / 'REPLACE-BG',
    'T1DiabetesGranada': DATA_ROOT / 'T1DiabetesGranada',
}

SENSOR_LIMITS = {
    'DIATREND':          (39.0, 401.0),
    'REPLACE-BG':        (39.0, 401.0),
    'T1DiabetesGranada': (40.0, 500.0),
}

FEATURE_COLS = [f'x{i}' for i in range(8)] + ['y']
KEY_COLS     = FEATURE_COLS + ['patient_id']

# Grupos de edad actualizados: <31, 31-45, 46-65, >=66
AGE_BINS   = [-np.inf, 30, 45, 65, np.inf]
AGE_LABELS = ['<31', '31-45', '46-65', '>=66']

# Patrón de nombre del archivo balanceado
BAL_PATTERN = re.compile(
    r'.*_fold(?P<fold>\d+)_(?P<group>age|sex)_(?P<technique>.+)\.parquet$'
)

# Clasificación de técnicas por comportamiento esperado
INCREASES_SIZE = {'oversampling', 'smote', 'jittering'}
DECREASES_SIZE = {'undersampling', 'patient_aware_undersampling', 'tomek_links'}
PRESERVES_SIZE = {'undersampling_oversampling', 'undersampling_smote', 'smote_tomek'}
CREATES_DUPS   = {'oversampling'}                      # ROS introduce duplicados exactos
NO_DUPS        = {'undersampling', 'patient_aware_undersampling', 'tomek_links'}  # nunca crea
SYNTHETIC      = {'smote', 'jittering', 'undersampling_smote', 'smote_tomek'}     # puntos nuevos

print('Rutas y constantes configuradas.')

Rutas y constantes configuradas.


In [7]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def load_original(dataset_dir: Path):
    candidates = sorted(dataset_dir.glob('windows_with_5folds_*.parquet'))
    if not candidates:
        raise FileNotFoundError(f'No se encontró archivo original en {dataset_dir}')
    df = pd.read_parquet(candidates[0])
    return df, candidates[0].name

def get_patient_info(dataset_dir: Path) -> pd.DataFrame:
    for cand in ['Patient_info.parquet', 'patient_info.parquet',
                 'Patient_info.csv',     'patient_info.csv']:
        p = dataset_dir / cand
        if p.exists():
            return pd.read_parquet(p) if p.suffix == '.parquet' else pd.read_csv(p)
    raise FileNotFoundError(f'No patient_info en {dataset_dir}')

def get_original_split(df_orig: pd.DataFrame, fold_idx: int, split: str) -> pd.DataFrame:
    col = f'fold_{fold_idx}'
    return df_orig[df_orig[col].str.lower() == split].reset_index(drop=True)

# ── Demografía ────────────────────────────────────────────────────────────────
def _find_col(df, candidates):
    norm = {c.strip().lower().replace(' ', '_'): c for c in df.columns}
    for cand in candidates:
        if cand.strip().lower().replace(' ', '_') in norm:
            return norm[cand.strip().lower().replace(' ', '_')]
    raise KeyError(f'No encontrado: {candidates}')

def add_age_group(df: pd.DataFrame, pid_col: str, pat_info: pd.DataFrame) -> pd.DataFrame:
    pi = pat_info.copy()
    pid_c = _find_col(pi, ['patient_id', 'Patient_ID', 'patientid'])
    pi['_pid'] = pi[pid_c].astype(str).str.strip()
    # Edad: columna Age o calcular desde Birth_year
    try:
        age_c = _find_col(pi, ['Age', 'age'])
        pi['_age'] = pd.to_numeric(pi[age_c], errors='coerce')
    except KeyError:
        birth_c = _find_col(pi, ['Birth_year', 'birth_year'])
        pi['_age'] = 2026 - pd.to_numeric(pi[birth_c], errors='coerce')
    pi['_age_group'] = pd.cut(
        pi['_age'], bins=AGE_BINS, labels=AGE_LABELS,
        include_lowest=True, right=True
    ).astype(str)
    lookup = pi.set_index('_pid')['_age_group'].to_dict()
    out = df.copy()
    out['age_group'] = df[pid_col].astype(str).str.strip().map(lookup).fillna('Unknown')
    return out

def add_sex_group(df: pd.DataFrame, pid_col: str, pat_info: pd.DataFrame) -> pd.DataFrame:
    pi = pat_info.copy()
    pid_c = _find_col(pi, ['patient_id', 'Patient_ID', 'patientid'])
    pi['_pid'] = pi[pid_c].astype(str).str.strip()
    sex_c = _find_col(pi, ['Sex', 'sex'])
    lookup = pi.set_index('_pid')[sex_c].str.upper().to_dict()
    out = df.copy()
    out['sex_group'] = df[pid_col].astype(str).str.strip().map(lookup).fillna('Unknown')
    return out

def add_demo_group(df, pid_col, pat_info, group):
    return add_age_group(df, pid_col, pat_info) if group == 'age' \
           else add_sex_group(df, pid_col, pat_info)

def group_col_name(group): return f'{group}_group'

# ── Estadísticos ──────────────────────────────────────────────────────────────
def describe_distribution(df, gcol, label):
    counts = df[gcol].value_counts(dropna=False).sort_index()
    pct    = (100 * counts / counts.sum()).round(1)
    result = pd.DataFrame({'n': counts, '%': pct})
    result.index.name = gcol
    result.columns = pd.MultiIndex.from_tuples([(label, 'n'), (label, '%')])
    return result

def count_exact_duplicates(df):
    cols = [c for c in FEATURE_COLS if c in df.columns]
    return int(df[cols].duplicated(keep='first').sum())

def count_synthetic(bal_train, orig_train):
    """Filas en bal_train cuya tupla de features NO existe en orig_train."""
    feat = [c for c in FEATURE_COLS if c in bal_train.columns and c in orig_train.columns]
    orig_set = set(map(tuple, orig_train[feat].round(4).values.tolist()))
    mask = ~bal_train[feat].round(4).apply(tuple, axis=1).isin(orig_set)
    return int(mask.sum()), mask

# ── Comprobaciones de integridad ──────────────────────────────────────────────
def check_val_test_integrity(orig_split, bal_split, split_name):
    if len(orig_split) != len(bal_split):
        return f'❌ {split_name}: tamaños distintos ({len(orig_split)} vs {len(bal_split)})'
    cols = [c for c in KEY_COLS if c in orig_split.columns and c in bal_split.columns]
    o = orig_split[cols].sort_values(cols).reset_index(drop=True)
    b = bal_split[cols].sort_values(cols).reset_index(drop=True)
    return (f'✅ {split_name}: idéntico al original ({len(orig_split):,} filas)'
            if o.equals(b) else
            f'⚠️  {split_name}: mismo nº de filas pero contenido distinto — revisar')

def check_no_patient_overlap(train_df, val_df, test_df):
    tp = set(train_df['patient_id'].astype(str))
    vp = set(val_df['patient_id'].astype(str))
    ep = set(test_df['patient_id'].astype(str))
    issues = []
    if tp & vp:  issues.append(f'train∩val={tp & vp}')
    if tp & ep:  issues.append(f'train∩test={tp & ep}')
    if vp & ep:  issues.append(f'val∩test={vp & ep}')
    return ('✅ Sin solapamiento de pacientes entre splits' if not issues
            else '❌ SOLAPAMIENTO: ' + ' | '.join(issues))

def check_sensor_range(df, synth_mask, sensor_min, sensor_max):
    feat = [c for c in FEATURE_COLS if c in df.columns]
    synth_df = df[synth_mask][feat]
    if len(synth_df) == 0:
        return '  (no hay filas sintéticas que comprobar)'
    out = ((synth_df < sensor_min) | (synth_df > sensor_max)).any(axis=1).sum()
    tag = '✅' if out == 0 else '❌'
    return (f'  {tag} Sintéticos fuera de rango [{sensor_min},{sensor_max}]: {out}  '
            f'| rango observado: [{synth_df.min().min():.2f}, {synth_df.max().max():.2f}]')

# ── Verificación de proporción equitativa (para técnicas combinadas) ──────────
def check_equal_proportions(bal_train, group_col, orig_total, tol=0.05):
    """
    Verifica que el total de train se preserva (±tol%) y que las proporciones
    entre grupos son iguales (±5pp). Devuelve lista de strings de resultado.
    """
    results = []
    bal_total = len(bal_train)
    size_diff = abs(bal_total - orig_total) / max(orig_total, 1)
    results.append(
        f'  {"✅" if size_diff <= tol else "⚠️"} Tamaño preservado: '
        f'original={orig_total:,}  balanceado={bal_total:,}  '
        f'(Δ={bal_total - orig_total:+,}, {size_diff*100:.1f}%)'
    )
    counts = bal_train[group_col].value_counts(dropna=False)
    groups = [g for g in counts.index if pd.notna(g) and g != 'Unknown']
    if len(groups) < 2:
        results.append('  ⚠️  Menos de 2 grupos con etiqueta conocida — no se puede verificar equidad')
        return results
    n_per_group = {g: counts.get(g, 0) for g in groups}
    pct_per_group = {g: 100 * n / bal_total for g, n in n_per_group.items()}
    expected_pct = 100 / len(groups)
    all_ok = all(abs(p - expected_pct) <= 5 for p in pct_per_group.values())
    results.append(
        f'  {"✅" if all_ok else "⚠️"} Proporciones esperadas ~{expected_pct:.1f}% por grupo:'
    )
    for g, n in sorted(n_per_group.items()):
        pct = pct_per_group[g]
        ok  = '✅' if abs(pct - expected_pct) <= 5 else '⚠️'
        results.append(f'    {ok} {g}: {n:,} ({pct:.1f}%)')
    return results

print('Helpers cargados.')

Helpers cargados.


In [8]:
# ── Función genérica de validación por archivo ────────────────────────────────

def validate_file(ds_name, df_orig, pat_info, bal_file, fold_idx, group, technique):
    """
    Valida un archivo balanceado concreto. Imprime resultados estructurados.
    Devuelve dict con métricas resumen para el cuadro global.
    """
    sensor_min, sensor_max = SENSOR_LIMITS[ds_name]
    gcol = group_col_name(group)

    df_bal    = pd.read_parquet(bal_file)
    bal_train = df_bal[df_bal['split'] == 'train'].reset_index(drop=True)
    bal_val   = df_bal[df_bal['split'] == 'val'].reset_index(drop=True)
    bal_test  = df_bal[df_bal['split'] == 'test'].reset_index(drop=True)

    orig_train = get_original_split(df_orig, fold_idx, 'train')
    orig_val   = get_original_split(df_orig, fold_idx, 'val')
    orig_test  = get_original_split(df_orig, fold_idx, 'test')

    orig_train_g = add_demo_group(orig_train, 'patient_id', pat_info, group)
    bal_train_g  = add_demo_group(bal_train,  'patient_id', pat_info, group)

    delta = len(bal_train) - len(orig_train)

    # ── Banner ────────────────────────────────────────────────────────────────
    print(f'\n{"═"*70}')
    print(f'  {ds_name} | fold={fold_idx} | grupo={group} | técnica={technique}')
    print(f'{"═"*70}')

    # ── 1. Tamaño de train ────────────────────────────────────────────────────
    print('\n── 1. TAMAÑO DE TRAIN')
    if technique in INCREASES_SIZE:
        size_ok = delta > 0
        tag = '(↑ aumento esperado ✅)' if size_ok else '(⚠️  no aumentó)'
    elif technique in DECREASES_SIZE:
        size_ok = delta <= 0
        tag = '(↓ reducción esperada ✅)' if size_ok else '(⚠️  no redujo)'
    elif technique in PRESERVES_SIZE:
        size_ok = abs(delta) / max(len(orig_train), 1) <= 0.05
        tag = '(≈ tamaño preservado ✅)' if size_ok else f'(⚠️  Δ={delta:+,})'
    else:
        size_ok = True; tag = '(—)'
    print(f'  Original  : {len(orig_train):>10,}')
    print(f'  Balanceado: {len(bal_train):>10,}')
    print(f'  Δ         : {delta:>+10,}  {tag}')

    # ── 2. Distribución de grupos ─────────────────────────────────────────────
    print(f'\n── 2. DISTRIBUCIÓN POR {gcol.upper()} EN TRAIN')
    dist_orig = describe_distribution(orig_train_g, gcol, 'Original')
    dist_bal  = describe_distribution(bal_train_g,  gcol, 'Balanceado')
    display(pd.concat([dist_orig, dist_bal], axis=1).fillna(0))

    # ── 3. Verificación específica según técnica ──────────────────────────────
    print(f'\n── 3. VERIFICACIÓN ESPECÍFICA [{technique}]')
    n_dup = count_exact_duplicates(bal_train)

    if technique == 'undersampling':
        orig_counts = orig_train_g[gcol].value_counts()
        expected_t  = int(orig_counts.min())
        bal_counts  = bal_train_g[gcol].value_counts()
        print(f'  Target esperado (min grupo original): {expected_t:,}')
        for g, n in sorted(bal_counts.items()):
            ok = '✅' if n <= expected_t + 1 else '⚠️'
            print(f'  {ok} {g}: {n:,}')
        print(f'  {"✅" if n_dup == 0 else "⚠️"}  Duplicados en train: {n_dup} (RUS no crea nuevos)')

    elif technique == 'oversampling':
        orig_counts = orig_train_g[gcol].value_counts()
        expected_t  = int(orig_counts.max())
        bal_counts  = bal_train_g[gcol].value_counts()
        print(f'  Target esperado (max grupo original): {expected_t:,}')
        for g, n in sorted(bal_counts.items()):
            ok = '✅' if n >= expected_t - 1 else '⚠️'
            print(f'  {ok} {g}: {n:,}')
        dup_ok = n_dup > 0
        print(f'  {"✅" if dup_ok else "⚠️"}  Duplicados en train: {n_dup} (ROS debe introducir réplicas)')

    elif technique == 'smote':
        n_synth, synth_mask = count_synthetic(bal_train, orig_train)
        print(f'  ✅ Filas sintéticas (no presentes en original): {n_synth:,}')
        print(f'  {"✅" if n_dup < 10 else "⚠️"}  Duplicados exactos (SMOTE interpola, no replica): {n_dup}')
        print(check_sensor_range(bal_train, synth_mask, sensor_min, sensor_max))

    elif technique == 'tomek_links':
        print(f'  Tomek Links elimina pares ruidosos en la frontera de decisión.')
        print(f'  No tiene target de proporción; solo limpia el contorno entre clases.')
        print(f'  {"✅" if delta <= 0 else "⚠️"}  Train reducido (se eliminaron pares Tomek): Δ={delta:+,}')
        print(f'  {"✅" if n_dup == 0 else "⚠️"}  Duplicados: {n_dup} (Tomek no crea)')
        # Verificar que ningún grupo aumentó en tamaño absoluto
        orig_counts = orig_train_g[gcol].value_counts()
        bal_counts  = bal_train_g[gcol].value_counts()
        print('  Tamaño por grupo (Tomek solo puede reducir):')
        for g in sorted(set(list(orig_counts.index) + list(bal_counts.index))):
            o = orig_counts.get(g, 0); b = bal_counts.get(g, 0)
            ok = '✅' if b <= o else '⚠️'
            print(f'    {ok} {g}: {o:,} → {b:,}  (Δ={b-o:+,})')

    elif technique == 'patient_aware_undersampling':
        # Todos los pacientes deben seguir presentes
        orig_pids = set(orig_train['patient_id'].astype(str))
        bal_pids  = set(bal_train['patient_id'].astype(str))
        missing   = orig_pids - bal_pids
        print(f'  {"✅" if not missing else "⚠️"}  Todos los pacientes conservados: '
              f'{len(bal_pids)}/{len(orig_pids)}'
              + (f' (faltan: {missing})' if missing else ''))
        print(f'  {"✅" if n_dup == 0 else "⚠️"}  Duplicados: {n_dup}')

    elif technique == 'jittering':
        n_synth, synth_mask = count_synthetic(bal_train, orig_train)
        print(f'  ✅ Filas con ruido añadido (nuevas): {n_synth:,}')
        print(f'  {"✅" if n_dup < 10 else "⚠️"}  Duplicados exactos (jitter no replica): {n_dup}')
        print(check_sensor_range(bal_train, synth_mask, sensor_min, sensor_max))

    elif technique in ('undersampling_oversampling', 'undersampling_smote', 'smote_tomek'):
        for line in check_equal_proportions(bal_train_g, gcol, len(orig_train)):
            print(line)
        if technique == 'smote_tomek':
            n_synth, synth_mask = count_synthetic(bal_train, orig_train)
            print(f'  ✅ Filas sintéticas (SMOTE): {n_synth:,}')
            print(check_sensor_range(bal_train, synth_mask, sensor_min, sensor_max))
        elif technique == 'undersampling_smote':
            n_synth, synth_mask = count_synthetic(bal_train, orig_train)
            print(f'  ✅ Filas sintéticas (SMOTE): {n_synth:,}')
            print(check_sensor_range(bal_train, synth_mask, sensor_min, sensor_max))
        elif technique == 'undersampling_oversampling':
            dup_comment = 'ROS puede introducir réplicas en grupos minoritarios'
            print(f'  ℹ️  Duplicados en train: {n_dup} ({dup_comment})')

    # ── 4. Integridad val y test ──────────────────────────────────────────────
    print('\n── 4. INTEGRIDAD DE VAL Y TEST')
    res_val  = check_val_test_integrity(orig_val,  bal_val,  'val')
    res_test = check_val_test_integrity(orig_test, bal_test, 'test')
    print(f'  {res_val}')
    print(f'  {res_test}')

    # ── 5. Solapamiento de pacientes ──────────────────────────────────────────
    print('\n── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS')
    print(f'  {check_no_patient_overlap(bal_train, bal_val, bal_test)}')

    val_ok     = res_val.startswith('✅')
    test_ok    = res_test.startswith('✅')
    overlap_ok = check_no_patient_overlap(bal_train, bal_val, bal_test).startswith('✅')

    if technique in INCREASES_SIZE:
        size_check = delta > 0
    elif technique in DECREASES_SIZE:
        size_check = delta <= 0
    elif technique in PRESERVES_SIZE:
        size_check = abs(delta) / max(len(orig_train), 1) <= 0.05
    else:
        size_check = True

    return {
        'Dataset':      ds_name,
        'Fold':         fold_idx,
        'Grupo':        group,
        'Técnica':      technique,
        'Orig. train':  f'{len(orig_train):,}',
        'Bal. train':   f'{len(bal_train):,}',
        'Δ filas':      f'{delta:+,}',
        'Δ OK':         '✅' if size_check else '❌',
        'Dup. train':   n_dup,
        'val intacto':  '✅' if val_ok   else '❌',
        'test intacto': '✅' if test_ok  else '❌',
        'Sin overlap':  '✅' if overlap_ok else '❌',
    }

print('Función de validación lista.')

Función de validación lista.


---
## Dataset 1: DIATREND

In [9]:
ds_name = 'DIATREND'
ds_dir  = DATASETS[ds_name]

df_orig_diatrend, orig_fname = load_original(ds_dir)
pat_info_diatrend = get_patient_info(ds_dir)

bal_files_diatrend = sorted((ds_dir / 'balanced_outputs').glob('windows_with_5folds_*.parquet'))
bal_files_diatrend = [f for f in bal_files_diatrend if BAL_PATTERN.match(f.name)]

print(f'Original : {orig_fname}  ({len(df_orig_diatrend):,} filas)')
print(f'Balanceados disponibles: {len(bal_files_diatrend)}')
for f in bal_files_diatrend:
    print(f'  {f.name}')

Original : windows_with_5folds_DiaTrend_2026-03-27_PH4.parquet  (2,304,689 filas)
Balanceados disponibles: 5
  windows_with_5folds_DiaTrend_2026-03-27_PH4_fold0_age_oversampling.parquet
  windows_with_5folds_DiaTrend_2026-03-27_PH4_fold1_sex_undersampling.parquet
  windows_with_5folds_DiaTrend_2026-03-27_PH4_fold2_age_tomek_links.parquet
  windows_with_5folds_DiaTrend_2026-03-27_PH4_fold3_sex_smote_tomek.parquet
  windows_with_5folds_DiaTrend_2026-03-27_PH4_fold4_age_smote.parquet


### DIATREND — Fold 0 · Age · Oversampling (ROS)

**Esperamos:** train mayor que el original (duplicados del grupo minoritario de edad), grupos igualados al máximo, val/test intactos.

In [10]:
rows_summary = []

fname = 'windows_with_5folds_DiaTrend_2026-03-27_PH4_fold0_age_oversampling.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('DIATREND', df_orig_diatrend, pat_info_diatrend,
                      bal_file, 0, 'age', 'oversampling')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  DIATREND | fold=0 | grupo=age | técnica=oversampling
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  1,598,896
  Balanceado:  6,056,508
  Δ         : +4,457,612  (↑ aumento esperado ✅)

── 2. DISTRIBUCIÓN POR AGE_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
age_group                                
31-45        20144   1.3    1514127  25.0
46-65        48725   3.0    1514127  25.0
<31        1514127  94.7    1514127  25.0
>=66         15900   1.0    1514127  25.0


── 3. VERIFICACIÓN ESPECÍFICA [oversampling]
  Target esperado (max grupo original): 1,514,127
  ✅ 31-45: 1,514,127
  ✅ 46-65: 1,514,127
  ✅ <31: 1,514,127
  ✅ >=66: 1,514,127
  ✅  Duplicados en train: 4459496 (ROS debe introducir réplicas)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (262,079 filas)
  ✅ test: idéntico al original (443,714 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### DIATREND — Fold 1 · Sex · Undersampling (RUS)

**Esperamos:** train menor que el original, sin duplicados nuevos, ambos sexos igualados al mínimo, val/test intactos.

In [11]:
fname = 'windows_with_5folds_DiaTrend_2026-03-27_PH4_fold1_sex_undersampling.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('DIATREND', df_orig_diatrend, pat_info_diatrend,
                      bal_file, 1, 'sex', 'undersampling')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  DIATREND | fold=1 | grupo=sex | técnica=undersampling
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  1,613,241
  Balanceado:    484,770
  Δ         : -1,128,471  (↓ reducción esperada ✅)

── 2. DISTRIBUCIÓN POR SEX_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
sex_group                                
F          1370856  85.0     242385  50.0
M           242385  15.0     242385  50.0


── 3. VERIFICACIÓN ESPECÍFICA [undersampling]
  Target esperado (min grupo original): 242,385
  ✅ F: 242,385
  ✅ M: 242,385
  ⚠️  Duplicados en train: 299 (RUS no crea nuevos)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (216,716 filas)
  ✅ test: idéntico al original (474,732 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### DIATREND — Fold 2 · Age · Tomek Links

**Esperamos:** train igual o ligeramente menor (solo se eliminan pares Tomek), sin objetivo de proporción específico (es limpieza de frontera), sin duplicados nuevos. Ningún grupo debe haber aumentado en tamaño absoluto.

In [12]:
fname = 'windows_with_5folds_DiaTrend_2026-03-27_PH4_fold2_age_tomek_links.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('DIATREND', df_orig_diatrend, pat_info_diatrend,
                      bal_file, 2, 'age', 'tomek_links')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  DIATREND | fold=2 | grupo=age | técnica=tomek_links
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  1,582,201
  Balanceado:  1,553,073
  Δ         :    -29,128  (↓ reducción esperada ✅)

── 2. DISTRIBUCIÓN POR AGE_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
age_group                                
31-45        12339   0.8      12339   0.8
46-65        44397   2.8      43919   2.8
<31        1509565  95.4    1480972  95.4
>=66         15900   1.0      15843   1.0


── 3. VERIFICACIÓN ESPECÍFICA [tomek_links]
  Tomek Links elimina pares ruidosos en la frontera de decisión.
  No tiene target de proporción; solo limpia el contorno entre clases.
  ✅  Train reducido (se eliminaron pares Tomek): Δ=-29,128
  ⚠️  Duplicados: 1296 (Tomek no crea)
  Tamaño por grupo (Tomek solo puede reducir):
    ✅ 31-45: 12,339 → 12,339  (Δ=+0)
    ✅ 46-65: 44,397 → 43,919  (Δ=-478)
    ✅ <31: 1,509,565 → 1,480,972  (Δ=-28,593)
    ✅ >=66: 15,900 → 15,843  (Δ=-57)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (258,508 filas)
  ✅ test: idéntico al original (463,980 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### DIATREND — Fold 3 · Sex · SMOTE + Tomek

**Esperamos:** tamaño similar al original (±5%), proporción equitativa entre sexos (~50/50), filas sintéticas dentro del rango del sensor, pocos o ningún duplicado exacto, val/test intactos.

In [13]:
fname = 'windows_with_5folds_DiaTrend_2026-03-27_PH4_fold3_sex_smote_tomek.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('DIATREND', df_orig_diatrend, pat_info_diatrend,
                      bal_file, 3, 'sex', 'smote_tomek')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  DIATREND | fold=3 | grupo=sex | técnica=smote_tomek
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  1,617,106
  Balanceado:  1,617,106
  Δ         :         +0  (≈ tamaño preservado ✅)

── 2. DISTRIBUCIÓN POR SEX_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
sex_group                                
F          1291094  79.8     808553  50.0
M           326012  20.2     808553  50.0


── 3. VERIFICACIÓN ESPECÍFICA [smote_tomek]
  ✅ Tamaño preservado: original=1,617,106  balanceado=1,617,106  (Δ=+0, 0.0%)
  ✅ Proporciones esperadas ~50.0% por grupo:
    ✅ F: 808,553 (50.0%)
    ✅ M: 808,553 (50.0%)
  ✅ Filas sintéticas (SMOTE): 482,475
  ✅ Sintéticos fuera de rango [39.0,401.0]: 0  | rango observado: [39.00, 401.00]

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (241,888 filas)
  ✅ test: idéntico al original (445,695 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### DIATREND — Fold 4 · Age · SMOTE

**Esperamos:** train mayor que el original, filas sintéticas por interpolación (no duplicados exactos), valores dentro del rango del sensor DiaTrend [39, 401] mg/dL, grupos igualados al máximo, val/test intactos.

In [14]:
fname = 'windows_with_5folds_DiaTrend_2026-03-27_PH4_fold4_age_smote.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('DIATREND', df_orig_diatrend, pat_info_diatrend,
                      bal_file, 4, 'age', 'smote')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  DIATREND | fold=4 | grupo=age | técnica=smote
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  1,604,177
  Balanceado:  4,652,544
  Δ         : +3,048,367  (↑ aumento esperado ✅)

── 2. DISTRIBUCIÓN POR AGE_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
age_group                                
31-45        28214   1.8    1550848  33.3
46-65        25115   1.6    1550848  33.3
<31        1550848  96.7    1550848  33.3


── 3. VERIFICACIÓN ESPECÍFICA [smote]
  ✅ Filas sintéticas (no presentes en original): 3,048,171
  ⚠️  Duplicados exactos (SMOTE interpola, no replica): 1511
  ✅ Sintéticos fuera de rango [39.0,401.0]: 0  | rango observado: [39.00, 401.00]

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (223,944 filas)
  ✅ test: idéntico al original (476,568 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


---
## Dataset 2: REPLACE-BG

In [15]:
ds_name = 'REPLACE-BG'
ds_dir  = DATASETS[ds_name]

df_orig_replace, orig_fname = load_original(ds_dir)
pat_info_replace = get_patient_info(ds_dir)

bal_files_replace = sorted((ds_dir / 'balanced_outputs').glob('windows_with_5folds_*.parquet'))
bal_files_replace = [f for f in bal_files_replace if BAL_PATTERN.match(f.name)]

print(f'Original : {orig_fname}  ({len(df_orig_replace):,} filas)')
print(f'Balanceados disponibles: {len(bal_files_replace)}')
for f in bal_files_replace:
    print(f'  {f.name}')

Original : windows_with_5folds_REPLACE-BG_2026-05-26_PH4.parquet  (3,305,411 filas)
Balanceados disponibles: 4
  windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold1_sex_patient_aware_undersampling.parquet
  windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold1_sex_undersampling_oversampling.parquet
  windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold2_age_undersampling.parquet
  windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold2_sex_undersampling_smote.parquet


### REPLACE-BG — Fold 1 · Sex · Patient-Aware Undersampling

**Esperamos:** train menor, sin duplicados nuevos, todos los pacientes del grupo mayoritario conservados (ningún paciente eliminado completamente), grupos igualados al mínimo.

In [16]:
fname = 'windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold1_sex_patient_aware_undersampling.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('REPLACE-BG', df_orig_replace, pat_info_replace,
                      bal_file, 1, 'sex', 'patient_aware_undersampling')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  REPLACE-BG | fold=1 | grupo=sex | técnica=patient_aware_undersampling
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  2,309,304
  Balanceado:  2,267,201
  Δ         :    -42,103  (↓ reducción esperada ✅)

── 2. DISTRIBUCIÓN POR SEX_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
sex_group                                
F          1151769  49.9    1151769  50.8
M          1157535  50.1    1115432  49.2


── 3. VERIFICACIÓN ESPECÍFICA [patient_aware_undersampling]
  ✅  Todos los pacientes conservados: 155/155
  ⚠️  Duplicados: 902

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (334,683 filas)
  ✅ test: idéntico al original (661,424 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### REPLACE-BG — Fold 1 · Sex · Undersampling + Oversampling

**Esperamos:** tamaño total similar al original (±5%), proporción ~50/50 entre sexos. Grupos mayoritarios reducidos con RUS y minoritarios ampliados con ROS. Pueden aparecer duplicados en el grupo que se amplió.

In [17]:
fname = 'windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold1_sex_undersampling_oversampling.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('REPLACE-BG', df_orig_replace, pat_info_replace,
                      bal_file, 1, 'sex', 'undersampling_oversampling')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  REPLACE-BG | fold=1 | grupo=sex | técnica=undersampling_oversampling
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  2,309,304
  Balanceado:  2,309,304
  Δ         :         +0  (≈ tamaño preservado ✅)

── 2. DISTRIBUCIÓN POR SEX_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
sex_group                                
F          1151769  49.9    1154652  50.0
M          1157535  50.1    1154652  50.0


── 3. VERIFICACIÓN ESPECÍFICA [undersampling_oversampling]
  ✅ Tamaño preservado: original=2,309,304  balanceado=2,309,304  (Δ=+0, 0.0%)
  ✅ Proporciones esperadas ~50.0% por grupo:
    ✅ F: 1,154,652 (50.0%)
    ✅ M: 1,154,652 (50.0%)
  ℹ️  Duplicados en train: 425780 (ROS puede introducir réplicas en grupos minoritarios)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (334,683 filas)
  ✅ test: idéntico al original (661,424 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### REPLACE-BG — Fold 2 · Age · Undersampling (RUS)

**Esperamos:** train reducido al tamaño del grupo minoritario de edad, sin duplicados nuevos, val/test intactos.

In [18]:
fname = 'windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold2_age_undersampling.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('REPLACE-BG', df_orig_replace, pat_info_replace,
                      bal_file, 2, 'age', 'undersampling')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  REPLACE-BG | fold=2 | grupo=age | técnica=undersampling
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  2,313,695
  Balanceado:    689,644
  Δ         : -1,624,051  (↓ reducción esperada ✅)

── 2. DISTRIBUCIÓN POR AGE_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
age_group                                
31-45       660555  28.5     172411  25.0
46-65       991536  42.9     172411  25.0
<31         489193  21.1     172411  25.0
>=66        172411   7.5     172411  25.0


── 3. VERIFICACIÓN ESPECÍFICA [undersampling]
  Target esperado (min grupo original): 172,411
  ✅ 31-45: 172,411
  ✅ 46-65: 172,411
  ✅ <31: 172,411
  ✅ >=66: 172,411
  ⚠️  Duplicados en train: 165 (RUS no crea nuevos)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (330,135 filas)
  ✅ test: idéntico al original (661,581 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### REPLACE-BG — Fold 2 · Sex · Undersampling + SMOTE

**Esperamos:** tamaño similar al original (±5%), proporción ~50/50 entre sexos, filas sintéticas (SMOTE) en el grupo minoritario dentro del rango del sensor [39, 401] mg/dL.

In [19]:
fname = 'windows_with_5folds_REPLACE-BG_2026-05-26_PH4_fold2_sex_undersampling_smote.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('REPLACE-BG', df_orig_replace, pat_info_replace,
                      bal_file, 2, 'sex', 'undersampling_smote')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  REPLACE-BG | fold=2 | grupo=sex | técnica=undersampling_smote
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  :  2,313,695
  Balanceado:  2,313,695
  Δ         :         +0  (≈ tamaño preservado ✅)

── 2. DISTRIBUCIÓN POR SEX_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
sex_group                                
F          1205205  52.1    1156848  50.0
M          1108490  47.9    1156847  50.0


── 3. VERIFICACIÓN ESPECÍFICA [undersampling_smote]
  ✅ Tamaño preservado: original=2,313,695  balanceado=2,313,695  (Δ=+0, 0.0%)
  ✅ Proporciones esperadas ~50.0% por grupo:
    ✅ F: 1,156,848 (50.0%)
    ✅ M: 1,156,847 (50.0%)
  ✅ Filas sintéticas (SMOTE): 48,346
  ✅ Sintéticos fuera de rango [39.0,401.0]: 0  | rango observado: [39.00, 401.00]

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (330,135 filas)
  ✅ test: idéntico al original (661,581 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


---
## Dataset 3: T1DiabetesGranada

In [20]:
ds_name = 'T1DiabetesGranada'
ds_dir  = DATASETS[ds_name]

df_orig_t1dg, orig_fname = load_original(ds_dir)
pat_info_t1dg = get_patient_info(ds_dir)

bal_files_t1dg = sorted((ds_dir / 'balanced_outputs').glob('windows_with_5folds_*.parquet'))
bal_files_t1dg = [f for f in bal_files_t1dg if BAL_PATTERN.match(f.name)]

print(f'Original : {orig_fname}  ({len(df_orig_t1dg):,} filas)')
print(f'Balanceados disponibles: {len(bal_files_t1dg)}')
for f in bal_files_t1dg:
    print(f'  {f.name}')

Original : windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4.parquet  (19,421,406 filas)
Balanceados disponibles: 3
  windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4_fold2_age_tomek_links.parquet
  windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4_fold4_age_smote.parquet
  windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4_fold4_sex_undersampling_oversampling.parquet


### T1DiabetesGranada — Fold 2 · Age · Tomek Links

**Esperamos:** limpieza de frontera sin objetivo de proporción, train igual o menor, sin duplicados nuevos. Grupos de edad `<31, 31-45, 46-65, >=66`.

In [21]:
fname = 'windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4_fold2_age_tomek_links.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('T1DiabetesGranada', df_orig_t1dg, pat_info_t1dg,
                      bal_file, 2, 'age', 'tomek_links')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  T1DiabetesGranada | fold=2 | grupo=age | técnica=tomek_links
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  : 13,597,404
  Balanceado: 11,525,487
  Δ         : -2,071,917  (↓ reducción esperada ✅)

── 2. DISTRIBUCIÓN POR AGE_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
age_group                                
31-45      4000179  29.4    3739259  32.4
46-65      5033371  37.0    3863413  33.5
<31        2886970  21.2    2465752  21.4
>=66       1676884  12.3    1457063  12.6


── 3. VERIFICACIÓN ESPECÍFICA [tomek_links]
  Tomek Links elimina pares ruidosos en la frontera de decisión.
  No tiene target de proporción; solo limpia el contorno entre clases.
  ✅  Train reducido (se eliminaron pares Tomek): Δ=-2,071,917
  ⚠️  Duplicados: 10180 (Tomek no crea)
  Tamaño por grupo (Tomek solo puede reducir):
    ✅ 31-45: 4,000,179 → 3,739,259  (Δ=-260,920)
    ✅ 46-65: 5,033,371 → 3,863,413  (Δ=-1,169,958)
    ✅ <31: 2,886,970 → 2,465,752  (Δ=-421,218)
    ✅ >=66: 1,676,884 → 1,457,063  (Δ=-219,821)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (1,934,027 filas)
  ✅ test: idéntico al original (3,889,975 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### T1DiabetesGranada — Fold 4 · Age · SMOTE

**Esperamos:** train mayor, sintéticos interpolados dentro de [40, 500] mg/dL, sin duplicados exactos, grupos de edad igualados al máximo.

In [22]:
fname = 'windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4_fold4_age_smote.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('T1DiabetesGranada', df_orig_t1dg, pat_info_t1dg,
                      bal_file, 4, 'age', 'smote')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  T1DiabetesGranada | fold=4 | grupo=age | técnica=smote
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  : 13,583,812
  Balanceado: 20,454,612
  Δ         : +6,870,800  (↑ aumento esperado ✅)

── 2. DISTRIBUCIÓN POR AGE_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
age_group                                
31-45      3918923  28.8    5158248  25.2
46-65      5545601  40.8    5854872  28.6
<31        2589517  19.1    3660947  17.9
>=66       1529771  11.3    5780545  28.3


── 3. VERIFICACIÓN ESPECÍFICA [smote]
  ✅ Filas sintéticas (no presentes en original): 6,869,283
  ⚠️  Duplicados exactos (SMOTE interpola, no replica): 22331
  ✅ Sintéticos fuera de rango [40.0,500.0]: 0  | rango observado: [40.00, 401.00]

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (1,943,190 filas)
  ✅ test: idéntico al original (3,894,404 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


### T1DiabetesGranada — Fold 4 · Sex · Undersampling + Oversampling

**Esperamos:** tamaño similar al original (±5%), proporción ~50/50 entre sexos, grupos mayoritarios reducidos con RUS y minoritarios ampliados con ROS.

In [23]:
fname = 'windows_with_5folds_T1DiabetesGranada_2026-05-26_PH4_fold4_sex_undersampling_oversampling.parquet'
bal_file = ds_dir / 'balanced_outputs' / fname
if bal_file.exists():
    r = validate_file('T1DiabetesGranada', df_orig_t1dg, pat_info_t1dg,
                      bal_file, 4, 'sex', 'undersampling_oversampling')
    rows_summary.append(r)
else:
    print(f'⚠️  Archivo no encontrado: {fname}')


══════════════════════════════════════════════════════════════════════
  T1DiabetesGranada | fold=4 | grupo=sex | técnica=undersampling_oversampling
══════════════════════════════════════════════════════════════════════

── 1. TAMAÑO DE TRAIN
  Original  : 13,583,812
  Balanceado: 13,583,812
  Δ         :         +0  (≈ tamaño preservado ✅)

── 2. DISTRIBUCIÓN POR SEX_GROUP EN TRAIN


Original       Balanceado      
                 n     %          n     %
sex_group                                
F          7409488  54.5    6791906  50.0
M          6174324  45.5    6791906  50.0


── 3. VERIFICACIÓN ESPECÍFICA [undersampling_oversampling]
  ✅ Tamaño preservado: original=13,583,812  balanceado=13,583,812  (Δ=+0, 0.0%)
  ✅ Proporciones esperadas ~50.0% por grupo:
    ✅ F: 6,791,906 (50.0%)
    ✅ M: 6,791,906 (50.0%)
  ℹ️  Duplicados en train: 2680737 (ROS puede introducir réplicas en grupos minoritarios)

── 4. INTEGRIDAD DE VAL Y TEST
  ✅ val: idéntico al original (1,943,190 filas)
  ✅ test: idéntico al original (3,894,404 filas)

── 5. SOLAPAMIENTO DE PACIENTES ENTRE SPLITS
  ✅ Sin solapamiento de pacientes entre splits


---
## Resumen global

In [24]:
if rows_summary:
    summary_df = pd.DataFrame(rows_summary)
    display(summary_df.set_index(['Dataset', 'Fold', 'Grupo', 'Técnica']))

    n_total = len(rows_summary)
    checks  = ['Δ OK', 'val intacto', 'test intacto', 'Sin overlap']
    print('\n── RESULTADOS GLOBALES')
    for check in checks:
        n_ok = (summary_df[check] == '✅').sum()
        print(f'  {check:20s}: {n_ok}/{n_total} ✅')
else:
    print('No hay resultados acumulados. Ejecuta las celdas anteriores primero.')

Orig. train  \
Dataset           Fold Grupo Técnica                                   
DIATREND          0    age   oversampling                  1,598,896   
                  1    sex   undersampling                 1,613,241   
                  2    age   tomek_links                   1,582,201   
                  3    sex   smote_tomek                   1,617,106   
                  4    age   smote                         1,604,177   
REPLACE-BG        1    sex   patient_aware_undersampling   2,309,304   
                             undersampling_oversampling    2,309,304   
                  2    age   undersampling                 2,313,695   
                       sex   undersampling_smote           2,313,695   
T1DiabetesGranada 2    age   tomek_links                  13,597,404   
                  4    age   smote                        13,583,812   
                       sex   undersampling_oversampling   13,583,812   

                                                          Bal. train  \
Dataset           Fold Grupo Técnica                                   
DIATREND          0    age   oversampling                  6,056,508   
                  1    sex   undersampling                   484,770   
                  2    age   tomek_links                   1,553,073   
                  3    sex   smote_tomek                   1,617,106   
                  4    age   smote                         4,652,544   
REPLACE-BG        1    sex   patient_aware_undersampling   2,267,201   
                             undersampling_oversampling    2,309,304   
                  2    age   undersampling                   689,644   
                       sex   undersampling_smote           2,313,695   
T1DiabetesGranada 2    age   tomek_links                  11,525,487   
                  4    age   smote                        20,454,612   
                       sex   undersampling_oversampling   13,583,812   

                                                             Δ filas Δ OK  \
Dataset           Fold Grupo Técnica                                        
DIATREND          0    age   oversampling                 +4,457,612    ✅   
                  1    sex   undersampling                -1,128,471    ✅   
                  2    age   tomek_links                     -29,128    ✅   
                  3    sex   smote_tomek                          +0    ✅   
                  4    age   smote                        +3,048,367    ✅   
REPLACE-BG        1    sex   patient_aware_undersampling     -42,103    ✅   
                             undersampling_oversampling           +0    ✅   
                  2    age   undersampling                -1,624,051    ✅   
                       sex   undersampling_smote                  +0    ✅   
T1DiabetesGranada 2    age   tomek_links                  -2,071,917    ✅   
                  4    age   smote                        +6,870,800    ✅   
                       sex   undersampling_oversampling           +0    ✅   

                                                          Dup. train  \
Dataset           Fold Grupo Técnica                                   
DIATREND          0    age   oversampling                    4459496   
                  1    sex   undersampling                       299   
                  2    age   tomek_links                        1296   
                  3    sex   smote_tomek                         772   
                  4    age   smote                              1511   
REPLACE-BG        1    sex   patient_aware_undersampling         902   
                             undersampling_oversampling       425780   
                  2    age   undersampling                       165   
                       sex   undersampling_smote                 952   
T1DiabetesGranada 2    age   tomek_links                       10180   
                  4    age   smote                             22331   
                       s


── RESULTADOS GLOBALES
  Δ OK                : 12/12 ✅
  val intacto         : 12/12 ✅
  test intacto        : 12/12 ✅
  Sin overlap         : 12/12 ✅


## Consistencia de val/test entre técnicas del mismo fold

Dos técnicas distintas sobre el mismo `(dataset, fold)` deben producir exactamente los mismos val y test.

In [25]:
from collections import defaultdict

by_fold = defaultdict(list)
for ds_name, ds_dir_ in DATASETS.items():
    bo = ds_dir_ / 'balanced_outputs'
    if bo.exists():
        for f in sorted(bo.glob('windows_with_5folds_*.parquet')):
            m = BAL_PATTERN.match(f.name)
            if m:
                by_fold[(ds_name, int(m.group('fold')))].append(f)

print('=== Consistencia de val/test entre técnicas del mismo (dataset, fold) ===\n')
any_multi = False
for (ds_name, fold_idx), files in sorted(by_fold.items()):
    if len(files) < 2:
        continue
    any_multi = True
    print(f'  {ds_name} — fold {fold_idx}  ({len(files)} técnicas):')
    ref_df   = pd.read_parquet(files[0])
    ref_val  = ref_df[ref_df['split'] == 'val'].reset_index(drop=True)
    ref_test = ref_df[ref_df['split'] == 'test'].reset_index(drop=True)
    for f in files[1:]:
        cmp_df   = pd.read_parquet(f)
        cmp_val  = cmp_df[cmp_df['split'] == 'val'].reset_index(drop=True)
        cmp_test = cmp_df[cmp_df['split'] == 'test'].reset_index(drop=True)
        rv = check_val_test_integrity(ref_val,  cmp_val,  'val' ).split(':')[0]
        rt = check_val_test_integrity(ref_test, cmp_test, 'test').split(':')[0]
        print(f'    {files[0].name.split("_fold")[1]}')
        print(f'    vs {f.name.split("_fold")[1]}')
        print(f'    val: {rv}  |  test: {rt}')
    print()

if not any_multi:
    print('  (No hay folds con más de un archivo balanceado disponible para comparar)')

=== Consistencia de val/test entre técnicas del mismo (dataset, fold) ===

  REPLACE-BG — fold 1  (2 técnicas):
    1_sex_patient_aware_undersampling.parquet
    vs 1_sex_undersampling_oversampling.parquet
    val: ✅ val  |  test: ✅ test

  REPLACE-BG — fold 2  (2 técnicas):
    2_age_undersampling.parquet
    vs 2_sex_undersampling_smote.parquet
    val: ✅ val  |  test: ✅ test

  T1DiabetesGranada — fold 4  (2 técnicas):
    4_age_smote.parquet
    vs 4_sex_undersampling_oversampling.parquet
    val: ✅ val  |  test: ✅ test



## Duplicados preexistentes en train original

Los duplicados observados en técnicas reductoras (undersampling, tomek_links) deben ser **iguales o menores** que los del train original del mismo fold, confirmando que no fueron introducidos por el balanceo.

In [26]:
print('=== Duplicados preexistentes en train original ===\n')

reduction_cases = [
    r for r in rows_summary
    if r['Técnica'] in (DECREASES_SIZE | {'undersampling_oversampling', 'undersampling_smote'})
    and r['Dup. train'] > 0
]

if not reduction_cases:
    print('  No hay casos reductores con duplicados para analizar.')
else:
    ds_orig_map = {
        'DIATREND':          df_orig_diatrend,
        'REPLACE-BG':        df_orig_replace,
        'T1DiabetesGranada': df_orig_t1dg,
    }
    for row in reduction_cases:
        orig_train = get_original_split(ds_orig_map[row['Dataset']], row['Fold'], 'train')
        n_dup_orig = count_exact_duplicates(orig_train)
        n_dup_bal  = row['Dup. train']
        ok = n_dup_bal <= n_dup_orig
        print(f'  {row["Dataset"]} fold{row["Fold"]} {row["Grupo"]} {row["Técnica"]}')
        print(f'    Dup. en orig. train : {n_dup_orig:,}')
        print(f'    Dup. en bal.  train : {n_dup_bal:,}')
        print(f'    {"✅ Preexistentes (balanceo no creó nuevos)" if ok else "⚠️  El balanceo introdujo duplicados nuevos — revisar"}')
        print()

=== Duplicados preexistentes en train original ===

  DIATREND fold1 sex undersampling
    Dup. en orig. train : 1,619
    Dup. en bal.  train : 299
    ✅ Preexistentes (balanceo no creó nuevos)

  DIATREND fold2 age tomek_links
    Dup. en orig. train : 1,296
    Dup. en bal.  train : 1,296
    ✅ Preexistentes (balanceo no creó nuevos)

  REPLACE-BG fold1 sex patient_aware_undersampling
    Dup. en orig. train : 916
    Dup. en bal.  train : 902
    ✅ Preexistentes (balanceo no creó nuevos)

  REPLACE-BG fold1 sex undersampling_oversampling
    Dup. en orig. train : 916
    Dup. en bal.  train : 425,780
    ⚠️  El balanceo introdujo duplicados nuevos — revisar

  REPLACE-BG fold2 age undersampling
    Dup. en orig. train : 958
    Dup. en bal.  train : 165
    ✅ Preexistentes (balanceo no creó nuevos)

  REPLACE-BG fold2 sex undersampling_smote
    Dup. en orig. train : 958
    Dup. en bal.  train : 952
    ✅ Preexistentes (balanceo no creó nuevos)

  T1DiabetesGranada fold2 age tomek